# First attempt at Implementing Quantum Adders

In [1]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT, UnitaryGate
from qiskit.transpiler import CouplingMap, generate_preset_pass_manager
from qiskit.visualization import plot_histogram

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [2]:
from dataclasses import dataclass

from qiskit import QuantumCircuit, QuantumRegister

@dataclass
class AdderResources:
    """Simple container for resource estimates of an n-bit adder.

    This is a first-order, approximate model corresponding to the
    naive ripple-carry adder implemented below.
    """
    n_bits: int
    toffoli_count: int
    cnot_count: int
    ancilla_count: int

In [3]:
def naive_ripple_adder_resources(n: int) -> AdderResources:
    """Return a rough resource estimate for the naive ripple-carry adder.

    Model:
      - Each bit uses ~3 Toffolis and ~3 CNOTs in its full-adder cell.
      - We allocate one carry ancilla per bit (in the `cout` register).

    This is not an optimized bound, but a simple, transparent starting point
    that matches the circuit structure implemented below.
    """
    toffoli_per_bit = 3
    cnot_per_bit = 3
    ancilla_per_bit = 1  # one carry ancilla in `cout`

    return AdderResources(
        n_bits=n,
        toffoli_count=toffoli_per_bit * n,
        cnot_count=cnot_per_bit * n,
        ancilla_count=ancilla_per_bit * n,
    )

In [4]:
def naive_full_adder_cell(
    circ: QuantumCircuit,
    x_qubit,
    y_qubit,
    cin_qubit,
    cout_qubit,
    s_qubit,
):
    """One-bit reversible full-adder cell.

    Conceptual behavior:
      - Inputs: x, y, cin, s=0, cout=0
      - Outputs:
          s    <- x ⊕ y ⊕ cin
          cout <- (x & y) ⊕ (x & cin) ⊕ (y & cin)

    We keep x, y, and cin unchanged. This is *not* an in-place adder;
    the sum is stored in a separate register `s`, and the carry is stored
    in `cout`.
    """
    # Sum bit into s: s = x ⊕ y ⊕ cin
    circ.cx(x_qubit, s_qubit)
    circ.cx(y_qubit, s_qubit)
    circ.cx(cin_qubit, s_qubit)

    # Carry-out into cout:
    # cout = (x & y) ⊕ (x & cin) ⊕ (y & cin)
    circ.ccx(x_qubit, y_qubit, cout_qubit)
    circ.ccx(x_qubit, cin_qubit, cout_qubit)
    circ.ccx(y_qubit, cin_qubit, cout_qubit)

In [5]:
def naive_ripple_adder_circuit(n: int) -> QuantumCircuit:
    """Construct a naive n-bit ripple-carry adder circuit (out-of-place).

    Registers (all size n except `cin`):
      - x:   input bits of the first addend
      - y:   input bits of the second addend (unchanged here)
      - cin: 1-qubit global carry-in (assumed initialized to |0>)
      - cout: n-qubit carry register (one ancilla per bit)
      - s:   n-qubit sum register (assumed initialized to |0...0>)

    Conceptual semantics:
      |x>|y>|0>_cin|0...0>_cout|0...0>_s
        → |x>|y>|0>_cin|carries>_cout|x+y>_s  (mod 2^n)

    This function does *not* perform any uncomputation of carries and
    is not optimized. It is intended as a first, transparent prototype
    of a ripple-carry quantum adder.
    """
    # Define registers
    x = QuantumRegister(n, "x")
    y = QuantumRegister(n, "y")
    cin = QuantumRegister(1, "cin")
    cout = QuantumRegister(n, "cout")
    s = QuantumRegister(n, "s")

    circ = QuantumCircuit(x, y, cin, cout, s)

    # For bit i:
    #   carry-in is cin[0] for i=0, otherwise cout[i-1]
    #   carry-out is cout[i]
    #   sum is written into s[i]
    for i in range(n):
        cin_qubit = cin[0] if i == 0 else cout[i - 1]
        cout_qubit = cout[i]
        s_qubit = s[i]

        naive_full_adder_cell(
            circ,
            x_qubit=x[i],
            y_qubit=y[i],
            cin_qubit=cin_qubit,
            cout_qubit=cout_qubit,
            s_qubit=s_qubit,
        )

    return circ

In [6]:
def getGateCount(circ):
    raw_counts = circ.count_ops()
    ops = {}
    for k, v in raw_counts.items():
        if isinstance(k, str):
            key = k
        else:
            # Instruction-like keys typically expose a 'name' attribute
            key = getattr(k, "name", str(k))
        ops[key] = v

    num_ccx = ops.get("ccx", 0)
    num_cx = ops.get("cx", 0)
    return num_ccx, num_cx

In [19]:
def run(n):
    circ = naive_ripple_adder_circuit(n)
    #print(circ)
    resources = naive_ripple_adder_resources(n)
    print("\nResource estimate (model):", resources)

    # Count actual gates in the constructed circuit
    # Normalize the keys from circ.count_ops() to string names so .get(...) accepts string keys
    num_ccx, num_cx = getGateCount(circ)
    print(f"Actual gate counts in circuit: ccx={num_ccx}, cx={num_cx}")
    return circ

In [ ]:
display(run(3))


Resource estimate (model): AdderResources(n_bits=3, toffoli_count=9, cnot_count=9, ancilla_count=3)
Actual gate counts in circuit: ccx=9, cx=9


None
